# Fase 1: Extracción de datos
---
Este cuaderno documenta la fase inicial de investigación y extracción de datos mediante web scraping. Se ha utilizado la discografía de Taylor Swift como **caso de estudio** para calibrar la conexión con la API de Genius.

El objetivo es ingestar el catálogo completo de la artista, estructurar la respuesta en formato JSON, evitar cuellos de botella mediante almacenamiento en caché local y consolidar los datos crudos en un DataFrame que servirá de base para el procesamiento del lenguaje natural en las fases posteriores.

### 1.1 Importación de librerías y variables de entorno
Se utiliza `python-dotenv` para cargar el token de la API de Genius de forma segura y evitar exponer credenciales en el código fuente.

In [2]:
import lyricsgenius
import pandas as pd
import os
import time
import json
from dotenv import load_dotenv, find_dotenv

### 1.2 Definición del catálogo de canciones
Se define el artista y la lista de álbumes de estudio para garantizar que el corpus textual tenga validez narrativa.

In [3]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

GENIUS_TOKEN = os.getenv("GENIUS_TOKEN")
ARTIST_NAME = "Taylor Swift"

ALBUMS = [
    "Taylor Swift",
    "Fearless (Taylor’s Version)",
    "Speak Now (Taylor’s Version)",
    "Red (Taylor’s Version)",
    "1989 (Taylor’s Version)",
    "reputation",
    "Lover",
    "folklore",
    "evermore",
    "Midnights (The Til Dawn Edition)",
    "Midnights (The Late Night Edition)",
    "THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY",
    "The Life Of A Showgirl"
]

In [4]:
genius = lyricsgenius.Genius(GENIUS_TOKEN, timeout=40, retries=3)

genius.verbose = True
genius.remove_section_headers = False
genius.skip_non_songs = True

all_songs_data = []

### 1.3 Extracción y caché local
El siguiente bucle itera sobre el catálogo y realiza peticiones a la API. Para optimizar el tiempo de ejecución y evitar baneos por *rate limiting*, el sistema guarda cada respuesta como un archivo `.json` en local. En ejecuciones posteriores, el script lee de disco en lugar de realizar llamadas a la red.

In [5]:
print(f"Iniciando descarga de {len(ALBUMS)} álbumes...\n")

for album_name in ALBUMS:

    safe_name = album_name.replace(" ", "_").replace(":", "").replace("'", "").replace("’", "").replace("(", "").replace(")", "")
    json_path = f"../data/raw/album_{safe_name}.json"

    if not os.path.exists(json_path):
        try:
            print(f"Descargando álbum: {album_name}...")
            album = genius.search_album(album_name, ARTIST_NAME)
            if album:
                album.save_lyrics(json_path)
            else:
                print(f"No se encontró el álbum: {album_name}")
        except Exception as e:
            print(f"Error al descargar {album_name}: {e}")
    else:
        print(f"Ya existe en disco: {album_name}. Saltando descarga.")

    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            songs_list = data.get('tracks', [])
            
            count = 0
            for item in songs_list:
                song_data = item.get('song') if 'song' in item else item

                if song_data:
                    if "Midnights" in album_name:
                        album_name = "Midnights"
                        
                    all_songs_data.append({
                        "title": song_data.get('title'),
                        "album": album_name,
                        "year": song_data.get('release_date_components', {}).get('year'),
                        "release_date": song_data.get('release_date_for_display'),
                        "lyrics": song_data.get('lyrics'),
                        "id": song_data.get('id'),
                        "url": song_data.get('url')
                    })
                    count += 1
            
            print(f"Procesadas {count} canciones de {album_name}.\n")

        except Exception as e:
            print(f"Error leyendo el JSON de {album_name}: {e}.")

    time.sleep(1)

Iniciando descarga de 13 álbumes...

Descargando álbum: Taylor Swift...
Searching for "Taylor Swift" by Taylor Swift...
Wrote ../data/raw/album_Taylor_Swift.json.
Procesadas 15 canciones de Taylor Swift.

Descargando álbum: Fearless (Taylor’s Version)...
Searching for "Fearless (Taylor’s Version)" by Taylor Swift...
Wrote ../data/raw/album_Fearless_Taylors_Version.json.
Procesadas 27 canciones de Fearless (Taylor’s Version).

Descargando álbum: Speak Now (Taylor’s Version)...
Searching for "Speak Now (Taylor’s Version)" by Taylor Swift...
Wrote ../data/raw/album_Speak_Now_Taylors_Version.json.
Procesadas 23 canciones de Speak Now (Taylor’s Version).

Descargando álbum: Red (Taylor’s Version)...
Searching for "Red (Taylor’s Version)" by Taylor Swift...
Wrote ../data/raw/album_Red_Taylors_Version.json.
Procesadas 31 canciones de Red (Taylor’s Version).

Descargando álbum: 1989 (Taylor’s Version)...
Searching for "1989 (Taylor’s Version)" by Taylor Swift...
Wrote ../data/raw/album_1989_Ta

### 1.4 Consolidación y limpieza de duplicados
En esta etapa se transforma la lista de diccionarios en un `pandas.DataFrame`. Se eliminan canciones duplicadas y se exporta la primera versión de la base de datos (`_discography.csv`).

In [ ]:
df = pd.DataFrame(all_songs_data)

if not df.empty:
    df.insert(0, 'artist', ARTIST_NAME)

    df = df.sort_values(by=['year', 'album'])

    print(f"\nCanciones totales cargadas: {len(df)}")

    df_clean = df.drop_duplicates(subset=['title'], keep='first')

    print(f"Canciones después de eliminar duplicados: {len(df_clean)}")
    print(f"Duplicados eliminados: {len(df) - len(df_clean)}")

    filename = ARTIST_NAME.lower().replace(" ", "_")
    csv_path = f"../data/interim/{filename}_discography.csv"
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df_clean.to_csv(csv_path, index=False)

    print("\n" + "="*50)
    print(f"Proceso completado.\nArchivo guardado en: {csv_path}")
    print("="*50)

    print(df.groupby('album')['title'].count())
else:
    print("El DataFrame está vacío.")


Canciones totales cargadas: 270
Canciones después de eliminar duplicados: 250
Duplicados eliminados: 20

Proceso completado.
Archivo guardado en: ../data/interim/taylor_swift_discography.csv
album
1989 (Taylor’s Version)                         22
Fearless (Taylor’s Version)                     27
Lover                                           18
Midnights                                       44
Red (Taylor’s Version)                          31
Speak Now (Taylor’s Version)                    23
THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY    31
Taylor Swift                                    15
The Life Of A Showgirl                          12
evermore                                        15
folklore                                        16
reputation                                      16
Name: title, dtype: int64
